In [1]:
from pathlib import Path
import gcamreader
import os
import pandas as pd
import numpy as np
from utils import convert_to_mt
import plotly.graph_objects as go

In [2]:
def to_Mt(row):
    val, unit = row['value'], row['Units']
    if unit == 'Tg':
        return val
    elif unit == 'Gg':
        return val * 1e-3
    elif unit == 'MTC':
        return val * (44.009 / 12.011)
    else:
        raise ValueError(f"Unknown unit: {unit}")

# AR5 100-yr GWP
GWP_AR5 = {
    'CO2':    1,
    'CH4':    28,
    'N2O':    265,
    'HFC125': 3500,
    'HFC134a':1430,
    'HFC143a':4470,
    'HFC23':  14800,
    'HFC32':  675,
    'HFC43':  1500,
    'HFC227ea':3220,
    'HFC236fa':9810,
    'SF6':    23500,
    'C2F6':   12200,
    'CF4':    6630,
}

In [3]:
dfCO2Map = pd.read_csv("./extdata/gcamreport/CO2_tech_map.csv", skiprows=[0])
dfCO2Map.head()

,sector,subsector,technology,var1,var2,var3,var4,var5,var6,var7,var8,var9,unit_conv
0,airCO2,airCO2,airCO2,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
1,CO2 removal,dac,hightemp DAC NG,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
2,CO2 removal,dac,hightemp DAC elec,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
3,CO2 removal,dac,lowtemp DAC heatpump,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
4,agricultural energy use,mobile,refined liquids,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Demand,Emissions|CO2|Energy|Demand|AFOFI,Emissions|CO2|Energy|Demand|Residential and Co...,NaN,NaN,NaN,3.666667


In [4]:
dfNonCO2Map = pd.read_csv("./extdata/gcamreport/nonCO2_emissions_sector_map.csv", skiprows=[0])
dfNonCO2Map

,sector,subsector,ghg,var1,var2,var3,var4,var5,var6,var7,var8,unit_conv
0,agricultural energy use,NaN,BC,Emissions|BC,Emissions|BC|Energy,Emissions|BC|Energy|Demand,Emissions|BC|Energy|Demand|AFOFI,NaN,NaN,Emissions|BC|Energy|Demand|Residential and Com...,Emissions|BC|Energy and Industrial Processes,1.0
1,agricultural energy use,NaN,CH4,Emissions|CH4,Emissions|CH4|Energy,Emissions|BC|Energy|Demand,Emissions|CH4|Energy|Demand|AFOFI,NaN,NaN,Emissions|CH4|Energy|Demand|Residential and Co...,Emissions|CH4|Energy and Industrial Processes,1.0
2,agricultural energy use,NaN,CO,Emissions|CO,Emissions|CO|Energy,Emissions|BC|Energy|Demand,Emissions|CO|Energy|Demand|AFOFI,NaN,NaN,Emissions|CO|Energy|Demand|Residential and Com...,Emissions|CO|Energy and Industrial Processes,1.0
3,agricultural energy use,NaN,N2O,Emissions|N2O,Emissions|N2O|Energy,Emissions|N2O|Energy|Demand,Emissions|N2O|Energy|Demand|AFOFI,NaN,NaN,Emissions|N2O|Energy|Demand|Residential and Co...,Emissions|N2O|Energy and Industrial Processes,1000.0
4,agricultural energy use,NaN,NH3,Emissions|NH3,Emissions|NH3|Energy,Emissions|NH3|Energy|Demand,Emissions|NH3|Energy|Demand|AFOFI,NaN,NaN,Emissions|NH3|Energy|Demand|Residential and Co...,Emissions|NH3|Energy and Industrial Processes,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
918,urban processes,NaN,OC,Emissions|OC,Emissions|OC|Other,NaN,NaN,NaN,NaN,NaN,NaN,1.0
919,urban processes,NaN,SO2_1,Emissions|Sulfur,Emissions|Sulfur|Other,NaN,NaN,NaN,NaN,NaN,NaN,1.0
920,urban processes,NaN,SO2_2,Emissions|Sulfur,Emissions|Sulfur|Other,NaN,NaN,NaN,NaN,NaN,NaN,1.0
921,urban processes,NaN,SO2_3,Emissions|Sulfur,Emissions|Sulfur|Other,NaN,NaN,NaN,NaN,NaN,NaN,1.0


In [5]:
def to_Mt(row):
    val, unit = row['value'], row['Units']
    if unit == 'Tg':
        return val
    elif unit == 'Gg':
        return val * 1e-3
    elif unit == 'MTC':
        return val * (44.009 / 12.011)
    else:
        raise ValueError(f"Unknown unit: {unit}")

# AR5 100-yr GWP
GWP_AR5 = {
    'CO2':    1,
    'CH4':    28,
    'CH4_AGR': 28,
    'CH4_AWB': 28,
    'N2O':    265,
    'N2O_AGR': 265,
    'N2O_AWB': 265,
    'HFC125': 3500,
    'HFC134a':1430,
    'HFC143a':4470,
    'HFC23':  14800,
    'HFC32':  675,
    'HFC43':  1500,
    'HFC227ea':3220,
    'HFC236fa':9810,
    'SF6':    23500,
    'C2F6':   12200,
    'CF4':    6630,
}

In [6]:
proj_path = Path("/data/project/tae/gcam-core")
xml_path = proj_path / "input" / "gcamdata" / "xml"
db_path = proj_path / "output"

In [8]:
dbpath = "../output/"  # relative to current working directory
dbfile = "database_basexdb_korea_2035_20250714"
conn = gcamreader.LocalDBConn(dbpath, dbfile)
queries = gcamreader.parse_batch_query(os.path.join('..', 'output', 'queries','Main_queries.xml'))

Database scenarios: Current-Policy, Enhanced-Ambition


In [9]:
scenarios = list(conn.listScenariosInDB()['name'])
scenarios

['Current-Policy', 'Enhanced-Ambition']

In [10]:
scenarios.reverse()

In [11]:
for i, q in enumerate(queries):
    print(i, q.title)

0 primary energy consumption by region (avg fossil efficiency)
1 primary energy consumption by region (direct equivalent)
2 primary energy consumption with CCS by region (direct equivalent)
3 resource production
4 resource production by tech and vintage
5 resource supply curves
6 regional primary energy prices
7 elec gen by region (incl CHP)
8 elec gen by subsector
9 elec gen by gen tech
10 elec gen by gen tech and cooling tech
11 elec gen by gen tech and cooling tech and vintage
12 elec gen by gen tech and cooling tech (new)
13 elec energy input by subsector
14 elec energy input by elec gen tech
15 elec energy input by elec gen tech and cooling tech
16 elec prices by sector
17 elec gen costs by subsector
18 elec gen costs by tech
19 elec gen costs by cooling tech
20 elec share-weights by subsector
21 elec share-weights by tech
22 elec share-weights by cooling tech
23 elec td inputs and outputs
24 cogeneration by region
25 elec consumption by demand sector
26 elec sector water withdraw

In [12]:
q = queries[262]
print(q.title)
dfCO2 = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
dfCO2['scenario'] = dfCO2['scenario'].str.split(',').str[0]
dfCO2['sector'] = dfCO2['sector'].str.replace(r'_d(?:[1-9]|10)$', '', regex=True)
dfCO2['GHG'] = 'CO2'

CO2 emissions by sector (no bio) (excluding resource production)


In [13]:
q = queries[272]
print(q.title)
dfNonCO2 = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
dfNonCO2['scenario'] = dfNonCO2['scenario'].str.split(',').str[0]
dfNonCO2['sector'] = dfNonCO2['sector'].str.replace(r'_d(?:[1-9]|10)$', '', regex=True)
dfNonCO2 = dfNonCO2[(dfNonCO2['GHG'].isin(GWP_AR5.keys()))]
dfNonCO2

nonCO2 emissions by subsector (excluding resource production)


,Units,scenario,region,sector,subsector,GHG,Year,value
0,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2005,0.253100
1,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2010,0.629604
2,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2015,0.717899
3,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2020,0.803929
4,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2025,0.787886
...,...,...,...,...,...,...,...,...
22998,Tg,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,N2O,2015,0.000123
22999,Tg,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,N2O,2020,0.000125
23000,Tg,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,N2O,2025,0.000123
23001,Tg,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,N2O,2030,0.000122


In [14]:
mask1 = ((dfNonCO2['sector'] == 'UnmanagedLand') & (dfNonCO2['subsector'].isin(['ForestFire', 'GrasslandFires'])))
mask2 = ((dfNonCO2['sector'] == 'urban processes') & (dfNonCO2['subsector'].isin(['landfills', 'wastewater', 'waste_incineration'])))

dfNonCO2_1 = dfNonCO2[~(mask1 | mask2)]
dfNonCO2_2 = dfNonCO2[mask1 | mask2]

print(dfNonCO2_1.shape, dfNonCO2_2.shape)

(6151, 8) (90, 8)


In [15]:
for sec in dfCO2['sector'].unique():
    if sec not in dfCO2Map['sector'].unique():
        print(sec)

electricity


In [16]:
for sec in dfNonCO2['sector'].unique():
    if sec not in dfNonCO2Map['sector'].unique():
        print(sec)

In [17]:
dfNonCO2[(dfNonCO2['sector'] == 'chemical feedstocks')]

,Units,scenario,region,sector,subsector,GHG,Year,value


In [18]:
dfCO2Map.head()

,sector,subsector,technology,var1,var2,var3,var4,var5,var6,var7,var8,var9,unit_conv
0,airCO2,airCO2,airCO2,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
1,CO2 removal,dac,hightemp DAC NG,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
2,CO2 removal,dac,hightemp DAC elec,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
3,CO2 removal,dac,lowtemp DAC heatpump,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
4,agricultural energy use,mobile,refined liquids,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Demand,Emissions|CO2|Energy|Demand|AFOFI,Emissions|CO2|Energy|Demand|Residential and Co...,NaN,NaN,NaN,3.666667


In [19]:
dfCO2Sec = dfCO2.merge(dfCO2Map[['sector', 'var1', 'var2', 'var3', 'var4', 'var5']].drop_duplicates(), on=['sector'], how='left')
dfCO2Sec.head()

,Units,scenario,region,sector,Year,value,GHG,var1,var2,var3,var4,var5
0,MTC,Current-Policy,South Korea,H2 central production,2020,0.002831,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen
1,MTC,Current-Policy,South Korea,H2 central production,2025,0.011654,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen
2,MTC,Current-Policy,South Korea,H2 central production,2030,0.004433,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen
3,MTC,Current-Policy,South Korea,H2 central production,2035,0.008392,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen
4,MTC,Current-Policy,South Korea,H2 wholesale dispensing,2020,0.002853,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen


In [20]:
dfNonCO2_1Sec = dfNonCO2_1.merge(dfNonCO2Map[(dfNonCO2Map['subsector'].isna())][['sector', 'ghg', 'var1', 'var2', 'var3', 'var4', 'var5']].drop_duplicates(), left_on=['sector', 'GHG'], right_on=['sector', 'ghg'], how='left')
dfNonCO2_1Sec.head()

,Units,scenario,region,sector,subsector,GHG,Year,value,ghg,var1,var2,var3,var4,var5
0,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2005,0.253100,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
1,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2010,0.629604,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
2,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2015,0.717899,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
3,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2020,0.803929,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
4,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2025,0.787886,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN


In [21]:
dfNonCO2_2Sec = dfNonCO2_2.merge(dfNonCO2Map[['sector', 'subsector', 'ghg', 'var1', 'var2', 'var3', 'var4', 'var5']].drop_duplicates(), left_on=['sector', 'subsector', 'GHG'], right_on=['sector', 'subsector', 'ghg'], how='left')
dfNonCO2_2Sec.head()

,Units,scenario,region,sector,subsector,GHG,Year,value,ghg,var1,var2,var3,var4,var5
0,Tg,Current-Policy,South Korea,urban processes,landfills,CH4,1975,0.013917,CH4,Emissions|CH4,Emissions|CH4|Waste,NaN,NaN,NaN
1,Tg,Current-Policy,South Korea,urban processes,landfills,CH4,1990,0.031836,CH4,Emissions|CH4,Emissions|CH4|Waste,NaN,NaN,NaN
2,Tg,Current-Policy,South Korea,urban processes,landfills,CH4,2005,0.042066,CH4,Emissions|CH4,Emissions|CH4|Waste,NaN,NaN,NaN
3,Tg,Current-Policy,South Korea,urban processes,landfills,CH4,2010,0.021372,CH4,Emissions|CH4,Emissions|CH4|Waste,NaN,NaN,NaN
4,Tg,Current-Policy,South Korea,urban processes,landfills,CH4,2015,0.021774,CH4,Emissions|CH4,Emissions|CH4|Waste,NaN,NaN,NaN


In [22]:
dfNonCO2_1.shape, dfNonCO2_1Sec.shape

((6151, 8), (6151, 14))

In [23]:
dfNonCO2_2.shape, dfNonCO2_2Sec.shape

((90, 8), (90, 14))

In [24]:
dfNonCO2Sec = pd.concat([dfNonCO2_1Sec, dfNonCO2_2Sec])
dfNonCO2Sec

,Units,scenario,region,sector,subsector,GHG,Year,value,ghg,var1,var2,var3,var4,var5
0,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2005,0.253100,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
1,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2010,0.629604,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
2,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2015,0.717899,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
3,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2020,0.803929,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
4,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2025,0.787886,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,Tg,Enhanced-Ambition,South Korea,urban processes,wastewater,N2O,2015,0.003289,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN
86,Tg,Enhanced-Ambition,South Korea,urban processes,wastewater,N2O,2020,0.003340,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN
87,Tg,Enhanced-Ambition,South Korea,urban processes,wastewater,N2O,2025,0.003312,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN
88,Tg,Enhanced-Ambition,South Korea,urban processes,wastewater,N2O,2030,0.002672,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN


In [25]:
def cat_sec(row):
    if row['sector'] == 'electricity':
        return 'Electricity'
    elif row['var2'].endswith('Other Capture and Removal'):
        return 'DAC'
    elif row['sector'] == 'cement':
        return 'Industry'
    elif row['sector'] == 'desalinated water':
        return 'Buildings'
    # elif row['var5'].endswith('Hydrogen'):
    #     return 'Hydrogen'
    elif row['var5'].endswith('Industry'):
        return 'Industry'
    elif row['var5'].endswith('Electricity'):
        return "Electricity"
    elif row['var5'].endswith('Residential and Commercial'):
        return "Buildings"
    elif row['var5'].endswith('Transportation'):
        return 'Transportation'
    elif row['sector'] in ['delivered biomass', 'delivered gas', 'gas pipeline', 'gas processing', 'refined liquids enduse', 'refined liquids industrial', 'refining', 'wholesale gas']:
        return 'Industry'
    elif row['var5'].endswith('AFOFI'):
        return 'Industry'
    else:
        print(row['sector'])
        return "Others"

In [26]:
dfCO2Sec['sec'] = dfCO2Sec.apply(cat_sec, axis=1)

H2 central production
H2 central production
H2 central production
H2 central production
H2 wholesale dispensing
H2 wholesale dispensing
H2 wholesale dispensing
H2 wholesale dispensing
H2 central production
H2 central production
H2 central production
H2 central production
H2 wholesale dispensing
H2 wholesale dispensing
H2 wholesale dispensing
H2 wholesale dispensing


In [27]:
dfNonCO2Sec[~(dfNonCO2Sec['var4'].isna())]['sector'].unique()

array(['industrial processes', 'Beef', 'Corn', 'Dairy', 'FiberCrop',
       'Fruits', 'H2 central production', 'Legumes', 'MiscCrop',
       'NutsSeeds', 'OilCrop', 'OtherGrain', 'Pork', 'Poultry', 'Rice',
       'RootTuber', 'SheepGoat', 'Soybean', 'UnmanagedLand', 'Vegetables',
       'Wheat', 'agricultural energy use', 'ammonia',
       'backup_electricity', 'biomass', 'chemical energy use',
       'comm cooling', 'comm heating', 'comm others',
       'construction energy use', 'electricity', 'mining energy use',
       'other industrial energy use', 'process heat cement',
       'process heat food processing', 'process heat paper', 'refining',
       'resid heating TradBio', 'resid heating coal',
       'resid heating modern', 'resid others TradBio',
       'resid others coal', 'resid others modern', 'trn_aviation_intl',
       'trn_freight', 'trn_freight_road', 'trn_pass', 'trn_pass_road',
       'trn_pass_road_LDV', 'trn_pass_road_LDV_4W', 'trn_shipping_intl'],
      dtype=object

In [28]:
dfNonCO2Sec[(dfNonCO2Sec['sector'] == 'urban processes') & ~(dfNonCO2Sec['GHG'].isin(['HFC125', 'HFC134a', 'HFC143a', 'HFC23', 'HFC32', 'HFC43', 'HFC227ea', 'HFC236fa', 'SF6', 'C2F6', 'CF4']))]['var2'].unique()

array(['Emissions|CH4|Waste', 'Emissions|N2O|Waste'], dtype=object)

In [29]:
def cat_sec_nonco2(row):
    if row['GHG'] in ['HFC125', 'HFC134a', 'HFC143a', 'HFC23', 'HFC32', 'HFC43', 'HFC227ea', 'HFC236fa', 'SF6', 'C2F6', 'CF4']:
        return 'F-Gases'
    # elif 'H2' in row['sector']:
    #     return 'Hydrogen'
    elif row['var2'].endswith("Waste"):
        return "Waste"
    # elif (row['var2'].endswith("AFOLU")) and ((row['var3'].endswith("Agriculture")) or (row['var3'].endswith('Agricultural Waste Burning'))):
    #     return 'Agriculture'
    elif (row['var2'].endswith("AFOLU")):
        return 'Others'
    elif (row['var2'].endswith('Industrial Processes')) or (row['sector'] == 'industrial processes'):
        return 'Industry'
    elif (row['var2'].endswith('Waste')) or (row['sector'] == 'urban processes'):
        return 'Others'#'Waste'
    elif row['var4'].endswith('Electricity'):
        return 'Electricity'
    elif (row['var2'].endswith('Industrial Processes')) or (row['var4'].endswith('Industry')):
        return 'Industry'
    elif row['var4'].endswith('Transportation'):
        return 'Transportation'
    # elif row['var4'].endswith('Hydrogen'):
    #     return 'Hydrogen' 
    elif row['var4'].endswith('Residential and Commercial'):
        return "Buildings"
    elif (row['var4'].endswith('AFOFI')) or (row['var4'].endswith('Heat')) or (row['var4'].endswith('Liquids')):
        return 'Industry'
    else:
        print(row['sector'])
        return "Others"

In [30]:
dfNonCO2Sec['sec'] = dfNonCO2Sec.apply(cat_sec_nonco2, axis=1)

H2 central production
H2 central production
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
H2 central production
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_int

In [31]:
dfGHGSec = pd.concat([dfCO2Sec, dfNonCO2Sec])
dfGHGSec

,Units,scenario,region,sector,Year,value,GHG,var1,var2,var3,var4,var5,sec,subsector,ghg
0,MTC,Current-Policy,South Korea,H2 central production,2020,0.002831,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN
1,MTC,Current-Policy,South Korea,H2 central production,2025,0.011654,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN
2,MTC,Current-Policy,South Korea,H2 central production,2030,0.004433,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN
3,MTC,Current-Policy,South Korea,H2 central production,2035,0.008392,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN
4,MTC,Current-Policy,South Korea,H2 wholesale dispensing,2020,0.002853,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,Tg,Enhanced-Ambition,South Korea,urban processes,2015,0.003289,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O
86,Tg,Enhanced-Ambition,South Korea,urban processes,2020,0.003340,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O
87,Tg,Enhanced-Ambition,South Korea,urban processes,2025,0.003312,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O
88,Tg,Enhanced-Ambition,South Korea,urban processes,2030,0.002672,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O


In [32]:
dfGHGSec['sec'].unique()

array(['Others', 'Industry', 'Electricity', 'Buildings', 'Transportation',
       'DAC', 'F-Gases', 'Waste'], dtype=object)

In [33]:
dfGHGSec[(dfGHGSec['sec'] == 'Agriculture')]['var3'].unique()

array([], dtype=object)

In [34]:
dfGHGSec['var3'].unique()

array(['Emissions|CO2|Energy', 'Emissions|CO2|Industrial Processes', nan,
       'Emissions|F-Gases', 'Emissions|CH4|AFOLU|Agriculture',
       'Emissions|N2O|AFOLU|Agriculture',
       'Emissions|CH4|AFOLU|Agricultural Waste Burning',
       'Emissions|N2O|AFOLU|Agricultural Waste Burning',
       'Emissions|N2O|AFOLU|Land', 'Emissions|CH4|Energy|Supply',
       'Emissions|CH4|AFOLU|Land', 'Emissions|BC|Energy|Demand',
       'Emissions|N2O|Energy|Demand',
       'Emissions|CH4|Industrial Processes|Chemicals',
       'Emissions|N2O|Industrial Processes|Chemicals',
       'Emissions|N2O|Energy|Supply', 'Emissions|CH4|Energy|Demand',
       'Emissions|CH4|Industrial Processes|Iron and Steel',
       'Emissions|N2O|Industrial Processes|Iron and Steel',
       'Emissions|CH4|Industrial Processes|Pulp and Paper',
       'Emissions|N2O|Industrial Processes|Pulp and Paper'], dtype=object)

In [35]:
dfGHGSec['emiss(MT)'] = dfGHGSec.apply(convert_to_mt, axis=1)
dfGHGSec['gwpAr5'] = dfGHGSec['GHG'].apply(lambda gas: GWP_AR5[gas] if gas in GWP_AR5 else np.nan)
dfGHGSec['MTCO2eq'] = dfGHGSec['emiss(MT)'] * dfGHGSec['gwpAr5']
dfGHGSec

,Units,scenario,region,sector,Year,value,GHG,var1,var2,var3,var4,var5,sec,subsector,ghg,emiss(MT),gwpAr5,MTCO2eq
0,MTC,Current-Policy,South Korea,H2 central production,2020,0.002831,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,0.010381,1,0.010381
1,MTC,Current-Policy,South Korea,H2 central production,2025,0.011654,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,0.042732,1,0.042732
2,MTC,Current-Policy,South Korea,H2 central production,2030,0.004433,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,0.016256,1,0.016256
3,MTC,Current-Policy,South Korea,H2 central production,2035,0.008392,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,0.030770,1,0.030770
4,MTC,Current-Policy,South Korea,H2 wholesale dispensing,2020,0.002853,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,0.010460,1,0.010460
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,Tg,Enhanced-Ambition,South Korea,urban processes,2015,0.003289,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O,0.003289,265,0.871601
86,Tg,Enhanced-Ambition,South Korea,urban processes,2020,0.003340,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O,0.003340,265,0.885222
87,Tg,Enhanced-Ambition,South Korea,urban processes,2025,0.003312,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O,0.003312,265,0.877770
88,Tg,Enhanced-Ambition,South Korea,urban processes,2030,0.002672,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O,0.002672,265,0.708207


In [36]:
dfGHGSec[(dfGHGSec['sector'] == 'Rice') & (dfGHGSec['Year'] == 2035)].groupby(['scenario', 'Year', 'subsector'])['MTCO2eq'].sum()

scenario           Year  subsector 
Current-Policy     2035  Rice_Korea    4.538229
Enhanced-Ambition  2035  Rice_Korea    3.755793
Name: MTCO2eq, dtype: float64

In [37]:
dfGHGSec[(dfGHGSec['sec'].isin(['Agriculture']))& (dfGHGSec['Year'] == 2035)].groupby(['scenario', 'Year', 'sector'])['MTCO2eq'].sum()

Series([], Name: MTCO2eq, dtype: float64)

In [38]:
dfGHGSec[(dfGHGSec['Year'].isin([2020, 2035])) & (dfGHGSec['sec'] == 'Waste')].groupby(['scenario', 'Year', 'subsector'])['MTCO2eq'].sum()

scenario           Year  subsector         
Current-Policy     2020  landfills              0.455641
                         waste_incineration     0.029051
                         wastewater            14.878558
                   2035  landfills              0.499192
                         waste_incineration     0.024775
                         wastewater            12.641603
Enhanced-Ambition  2020  landfills              0.455641
                         waste_incineration     0.029051
                         wastewater            14.878558
                   2035  landfills              0.313642
                         waste_incineration     0.020612
                         wastewater            10.518999
Name: MTCO2eq, dtype: float64

In [39]:
dfGHGSec.groupby(['scenario', 'Year'])['MTCO2eq'].sum()

scenario           Year
Current-Policy     1975     54.168502
                   1990    296.523004
                   2005    601.039565
                   2010    698.621279
                   2015    747.006066
                   2020    719.373163
                   2025    699.351082
                   2030    632.503105
                   2035    551.264073
Enhanced-Ambition  1975     54.168502
                   1990    296.523004
                   2005    601.039565
                   2010    698.621279
                   2015    747.006066
                   2020    719.373163
                   2025    699.247924
                   2030    553.532755
                   2035    412.267974
Name: MTCO2eq, dtype: float64

In [40]:
dfGHGSec[(dfGHGSec['sector'].str.contains('H2')) & (~dfGHGSec['sector'].isin(['trn_aviation_intl', 'trn_shipping_intl'])) & (dfGHGSec['Year'] == 2035)]#['var4'].unique()

,Units,scenario,region,sector,Year,value,GHG,var1,var2,var3,var4,var5,sec,subsector,ghg,emiss(MT),gwpAr5,MTCO2eq
3,MTC,Current-Policy,South Korea,H2 central production,2035,8.391781e-03,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,3.076986e-02,1,3.076986e-02
7,MTC,Current-Policy,South Korea,H2 wholesale dispensing,2035,6.669891e-02,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,2.445627e-01,1,2.445627e-01
594,MTC,Enhanced-Ambition,South Korea,H2 central production,2035,1.967022e-03,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,7.212415e-03,1,7.212415e-03
598,MTC,Enhanced-Ambition,South Korea,H2 wholesale dispensing,2035,8.240289e-03,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,3.021439e-02,1,3.021439e-02
1106,Tg,Current-Policy,South Korea,H2 central production,2035,2.422490e-09,CH4,Emissions|CH4,Emissions|CH4|Energy,Emissions|CH4|Energy|Supply,Emissions|CH4|Energy|Supply|Hydrogen,NaN,Others,biomass,CH4,2.422490e-09,28,6.782972e-08


In [41]:
dfGHGSec[(dfGHGSec['sec'] == 'Others') & (~dfGHGSec['sector'].isin(['trn_aviation_intl', 'trn_shipping_intl'])) & (dfGHGSec['Year'] == 2035)]#['var4'].unique()

,Units,scenario,region,sector,Year,value,GHG,var1,var2,var3,var4,var5,sec,subsector,ghg,emiss(MT),gwpAr5,MTCO2eq
3,MTC,Current-Policy,South Korea,H2 central production,2035,8.391781e-03,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,3.076986e-02,1,0.030770
7,MTC,Current-Policy,South Korea,H2 wholesale dispensing,2035,6.669891e-02,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,2.445627e-01,1,0.244563
594,MTC,Enhanced-Ambition,South Korea,H2 central production,2035,1.967022e-03,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,7.212415e-03,1,0.007212
598,MTC,Enhanced-Ambition,South Korea,H2 wholesale dispensing,2035,8.240289e-03,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,3.021439e-02,1,0.030214
924,Tg,Current-Policy,South Korea,Beef,2035,7.607879e-02,CH4_AGR,Emissions|CH4,Emissions|CH4|AFOLU,Emissions|CH4|AFOLU|Agriculture,Emissions|CH4|AFOLU|Agriculture|Livestock,Emissions|CH4|AFOLU|Agriculture|Livestock|Ente...,Others,Mixed,CH4_AGR,7.607879e-02,28,2.130206
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4140,Tg,Enhanced-Ambition,South Korea,Wheat,2035,3.964017e-06,CH4_AWB,Emissions|CH4,Emissions|CH4|AFOLU,Emissions|CH4|AFOLU|Agricultural Waste Burning,NaN,NaN,Others,Wheat_Korea,CH4_AWB,3.964017e-06,28,0.000111
4149,Tg,Enhanced-Ambition,South Korea,Wheat,2035,1.647207e-04,N2O_AGR,Emissions|N2O,Emissions|N2O|AFOLU,Emissions|N2O|AFOLU|Agriculture,Emissions|N2O|AFOLU|Agriculture|Managed Soils,NaN,Others,Wheat_Korea,N2O_AGR,1.647207e-04,265,0.043651
4158,Tg,Enhanced-Ambition,South Korea,Wheat,2035,6.809260e-08,N2O_AWB,Emissions|N2O,Emissions|N2O|AFOLU,Emissions|N2O|AFOLU|Agricultural Waste Burning,NaN,NaN,Others,Wheat_Korea,N2O_AWB,6.809260e-08,265,0.000018
4224,Tg,Enhanced-Ambition,South Korea,biomass,2035,1.181980e-05,N2O_AGR,Emissions|N2O,Emissions|N2O|AFOLU,Emissions|N2O|AFOLU|Agriculture,Emissions|N2O|AFOLU|Agriculture|Managed Soils,NaN,Others,biomassGrass_Korea,N2O_AGR,1.181980e-05,265,0.003132


In [42]:
dfGHGSecDiff = dfGHGSec[(dfGHGSec['Year'].isin([2020, 2035]) & (~dfGHGSec['sector'].isin(['trn_aviation_intl', 'trn_shipping_intl']))) & (dfGHGSec['scenario'].isin(['Current-Policy', 'Enhanced-Ambition', 'Enhanced-Ambition-Bld']))].groupby(['scenario', 'Year', 'sec'])['MTCO2eq'].sum().reset_index().pivot(index=['scenario', 'sec'], columns=['Year'], values='MTCO2eq')
dfGHGSecDiff

Year                                    2020        2035
scenario          sec                                   
Current-Policy    Buildings        50.258276   48.506059
                  Electricity     243.299908  118.003636
                  F-Gases          42.735985   45.103919
                  Industry        206.635628  181.782078
                  Others           16.491352   17.888043
                  Transportation  118.202630   99.994269
                  Waste            15.363250   13.165570
Enhanced-Ambition Buildings        50.258276   35.657092
                  DAC                    NaN   -2.959733
                  Electricity     243.299908   55.264711
                  F-Gases          42.735985   38.043630
                  Industry        206.635628  143.880565
                  Others           16.491352   15.761703
                  Transportation  118.202630   89.318634
                  Waste            15.363250   10.853253

In [43]:
dfGHGSec[(~dfGHGSec['sector'].isin(['trn_aviation_intl', 'trn_shipping_intl']))].groupby(['Year', 'scenario'])['MTCO2eq'].sum()

Year  scenario         
1975  Current-Policy        53.918215
      Enhanced-Ambition     53.918215
1990  Current-Policy       291.119683
      Enhanced-Ambition    291.119683
2005  Current-Policy       569.489759
      Enhanced-Ambition    569.489759
2010  Current-Policy       671.273365
      Enhanced-Ambition    671.273365
2015  Current-Policy       720.820835
      Enhanced-Ambition    720.820835
2020  Current-Policy       692.987029
      Enhanced-Ambition    692.987029
2025  Current-Policy       672.443314
      Enhanced-Ambition    672.340155
2030  Current-Policy       605.906060
      Enhanced-Ambition    527.326535
2035  Current-Policy       524.443574
      Enhanced-Ambition    385.819855
Name: MTCO2eq, dtype: float64

In [44]:
dfGHGSecDiff.fillna(0, inplace=True)
dfGHGSecDiff['Diff'] = dfGHGSecDiff[2035] - dfGHGSecDiff[2020]
dfGHGSecDiff

Year                                    2020        2035        Diff
scenario          sec                                               
Current-Policy    Buildings        50.258276   48.506059   -1.752217
                  Electricity     243.299908  118.003636 -125.296272
                  F-Gases          42.735985   45.103919    2.367934
                  Industry        206.635628  181.782078  -24.853549
                  Others           16.491352   17.888043    1.396691
                  Transportation  118.202630   99.994269  -18.208361
                  Waste            15.363250   13.165570   -2.197680
Enhanced-Ambition Buildings        50.258276   35.657092  -14.601184
                  DAC               0.000000   -2.959733   -2.959733
                  Electricity     243.299908   55.264711 -188.035197
                  F-Gases          42.735985   38.043630   -4.692355
                  Industry        206.635628  143.880565  -62.755063
                  Others           16.491352   15.761703   -0.729649
                  Transportation  118.202630   89.318634  -28.883995
                  Waste            15.363250   10.853253   -4.509998

In [58]:
emiss_2018 = 783.8
emiss_2020 = 638.41
power_base=emiss_2020
power_ep = 	-188.03
power_cp = -125.29
ind_base = emiss_2020 + power_ep
ind_ep = -62.75
ind_cp = -24.85
trn_base = ind_base + ind_ep
trn_ep = -28.88
trn_cp = -18.2
bld_base = trn_base + trn_ep
bld_ep = -14.60
bld_cp = -1.75
waste_base = bld_base + bld_ep
waste_ep = -4.50
waste_cp = -2.19
fgas_base = waste_base + waste_ep
fgas_ep = -4.7
fgas_cp = 0

dac_base = fgas_base + fgas_ep
dac_ep = -2.95
dac_cp = 0

lulucf_ep = -9
lulucf_cp = 0
lulucf_base = dac_base + dac_ep

emiss_2035_ep = 322.5


power_2018 = 278.85
ind_2018 = 279.97
trn_2018 = 98.76
bld_2018 = 48.90
waste_2018 = 19.3
fgas_2018 = 23.59

rr_power = - power_ep / power_2018 * 100
rr_ind = - ind_ep / ind_2018 * 100
rr_trn = -trn_ep / trn_2018 * 100
rr_bld = -bld_ep / bld_2018 * 100
rr_waste = -waste_ep / waste_2018 * 100
rr_fgas = -fgas_ep / fgas_2018 * 100

In [59]:
ind_base

450.38

In [61]:
rd_ttl = emiss_2035_ep - emiss_2018
rr_ttl = -rd_ttl / emiss_2018 * 100

In [62]:
data = pd.DataFrame({
    "category": [
        "2018",
        "2020",
        "Power", "Industry", "Transportation", "Buildings", "Waste", "F-Gases", "DAC", "LULUCF", 
        "2035"
    ],
})

fig = go.Figure()

# Start and end bars
fig.add_trace(go.Waterfall(
    name="Enhanced Ambition",
    orientation="v",
    measure=["absolute"] * 2 + ["relative"] * 8 + ["total"],
    x=data["category"],
    y=[emiss_2018, emiss_2020, power_ep, ind_ep, trn_ep, bld_ep, waste_ep, fgas_ep, dac_ep, lulucf_ep, emiss_2035_ep],
    base=0,
    connector={"visible": False},
    decreasing={"marker": {"color": "#1f77b4"}},   # enhanced ambition
    increasing={"marker": {"color": "#FF6692"}},   # current policies
    totals={"marker": {"color": "lightgray"}},
    showlegend=False
))


# Add Current Policy overlays just for Coal categories
fig.add_trace(go.Bar(
    name="Current Policy",
    x=["Power", "Industry", "Transportation", "Buildings", "Waste", "F-Gases", "DAC", "LULUCF"],
    y=[power_cp, ind_cp, trn_cp, bld_cp, waste_cp, fgas_cp, dac_cp, lulucf_cp],  # smaller reductions
    base=[power_base, ind_base, trn_base, bld_base, waste_base, fgas_base, dac_base, lulucf_base],  # position on top of previous waterfall step
    marker_color="#AEC7E8",
    showlegend=False
))

fig.update_layout(
    width=950,
    height=500,
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white',
    title="<b>Emissions Reductions from Each Sector Compared to 2018 Levels</b>",
    title_font_size=21,
    title_x=0.5,

)

# Add dummy scatter trace to label bar values at center
fig.add_trace(go.Scatter(
    x=["2018", "2020", "2035"],
    y=[emiss_2020 / 2, emiss_2020 / 2, emiss_2035_ep / 2],
    mode="text",
    text=[f"<b>{emiss_2018:.1f}<br>TOTAL</b>", f"<b>{emiss_2020:.1f}<br>NET</b>", f"<b>{emiss_2035_ep:.1f}<br>NET</b>"],
    textposition="middle center",
    showlegend=False
))

# Add dummy scatter trace to label bar values at center
fig.add_trace(go.Scatter(
    x=["Power", "Industry", "Transportation", "Buildings", "Waste", "F-Gases", "DAC", "LULUCF"],
    #-190.5, -73.8, -28.9, -15.9, -4.5, -4.9, -3.0, -9
    y=[x+40 for x in [power_base, ind_base, trn_base, bld_base, waste_base, fgas_base]] + [y + 10 for y in [dac_base, lulucf_base]],
    mode="text",
    text=[f"<b>{power_ep:.1f}<br>(△{rr_power:.1f}%)</b>", f"<b>{ind_ep:.1f}<br>(△{rr_ind:.1f}%)</b>", f"<b>{trn_ep:.1f}<br>(△{rr_trn:.1f}%)</b>", 
          f"<b>{bld_ep:.1f}<br>(△{rr_bld:.1f}%)</b>", f"<b>{waste_ep:.1f}<br>(△{rr_waste:.1f}%)</b>", f"<b>{fgas_ep:.1f}<br>(△{rr_fgas:.1f}%)</b>",
          f"<b>{dac_ep:.1f}</b>", f"<b>{lulucf_ep:.1f}</b>"],
    textposition="middle center",
    showlegend=False
))

fig.add_trace(go.Bar(
    x=[None], y=[None],
    name="Current Policy",
    marker=dict(color="#AEC7E8"),
    showlegend=True,
    hoverinfo="skip"
))

fig.add_trace(go.Bar(
    x=[None], y=[None],
    name="Enhanced Ambition",
    marker=dict(color="#1f77b4"),
    showlegend=True,
    hoverinfo="skip"
))


fig.update_layout(
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.3,
        xanchor='center',
        x=0.5,
        bgcolor='rgba(0,0,0,0)',
        borderwidth=0,
        font=dict(size=15)
    )
)

fig.update_layout(
    yaxis=dict(
        showgrid=True,
        gridcolor='lightgrey',
        title="Emission (MtCO2e)", title_font_size=18,
        tickvals=list(range(0, 801, 100)),
    )
)

fig.add_annotation(
    x=10, y=emiss_2035_ep + 5,        # 7 is the index of "2035" in the x-category list
    ax=10, ay=emiss_2018 + 30,
    xref="x", yref="y",
    axref="x", ayref="y",
    text=f"<b>{rd_ttl:.1f}<br>(△{rr_ttl:.1f}%)</b>",
    showarrow=True,
    arrowhead=3,
    arrowwidth=3,
    arrowsize=1,
    arrowcolor="#1f77b4",
    font=dict(size=12, color="black"),
    align="center"
)

fig.add_annotation(
    x=2 + 0.2, y=ind_base + 5,        # 7 is the index of "2035" in the x-category list
    ax=2 + 0.2, ay=emiss_2020,
    xref="x", yref="y",
    axref="x", ayref="y",
    # text="<b>Enhanced<br>Ambition</b>",
    showarrow=True,
    arrowhead=1,
    arrowwidth=3,
    arrowsize=1,
    arrowcolor="#00A08B",
    font=dict(size=10, color="#00A08B"),
    align="center"
)

fig.add_annotation(
    x=2 -0.2, y=emiss_2020 + power_cp + 5,        # 7 is the index of "2035" in the x-category list
    ax=2 - 0.2, ay=emiss_2020,
    xref="x", yref="y",
    axref="x", ayref="y",
    # text="<b>CoalOut</b>",
    showarrow=True,
    arrowhead=1,
    arrowwidth=3,
    arrowsize=1,
    arrowcolor="#1616A7",
    font=dict(size=10, color="#1616A7"),
    align="center"
)


fig.update_layout(
    xaxis=dict(
        tickvals=list(range(len(data['category']))),
        ticktext=[f"<b>{cat}</b>" for cat in data['category']],
        # tickfont=dict(size=15)
    )
)

fig.add_annotation(
    text="<b>Current<br>Policy</b>",
    # xref="paper", yref="paper",
    x=2-0.6, y=570,
    showarrow=False,
    font=dict(size=10, color="#1616A7"),
    align="left"
)

fig.add_annotation(
    text="<b>Enhanced<br>Ambition</b>",
    # xref="paper", yref="paper",
    x=2+0.65, y=570,
    showarrow=False,
    font=dict(size=10, color="#00A08B"),
    align="left"
)

fig